# Coupled null-baseline probe

- **Project:** Opinion-dynamics baseline model
- **Submodel ID:** `coupled-null-baseline-v0.1`
- **Framework context:** `opinion-model` 0.1.0 working baseline
- **Parent probes:** `single_agent_information_effect_probe.ipynb`; `beta_learning_rate_initialization_probe.ipynb`
- **Probe type:** coupled mechanism probe under a null interaction boundary
- **Date and owner:** 2026-09-03; researcher-led, assistant-implemented
- **Status:** runnable package-backed draft for researcher review
- **Highest intended claim level:** V2 (paired coupling under synthetic boundary conditions)

This notebook closes the smallest population loop needed to connect message production, neutral exposure, homogeneous evidence aggregation, and Beta-belief updating. The scientific implementation is imported from `opinion_model`; this notebook owns configuration, controlled checks, visualization, and conditional interpretation. It is a diagnostic control model, not an empirical representation of a platform or population.


## 1. Question, decision, and framework link

**Primary question.** Can the extracted message-production, selection, aggregation, opinion-update, static-network, and scheduling components form one reproducible, order-invariant population loop under an explicitly neutral interaction boundary?

**Decision this probe supports.** Decide whether this package-backed loop is clear and stable enough to retain as the shared null benchmark before either scenario substitutes a mechanism.

**Researcher-confirmed boundary.** The probe uses 11 agents and 10 rounds. Every agent receives one production opportunity per round, produces one post under the current baseline probability of one, and consumes the ten posts produced by the other agents.

**Shared states and interfaces.** The private state is `BetaBelief(a, b)`; a `ProductionOutcome` records post/no-post; the observable message stance is $M\in\{-1,+1\}$; an `Exposure` links one message to one consumer; `MessageEvidence` preserves raw and weighted aggregates; all opinion and network changes are proposals until synchronous commit.

A direct calculation can verify one update but cannot verify the coupled schedule, self-exclusion, exposure equality, independent random streams, event reconciliation, replaceable component interfaces, or simultaneous commit across agents.


## 2. Provenance and boundary contract

| Element | Form used here | Provenance and decision status |
|---|---|---|
| Population and duration | 11 agents, 10 rounds | Researcher-confirmed for this toy experiment |
| Private state | $P_{i,t}\sim\mathrm{Beta}(a_{i,t},b_{i,t})$ | Retained from the parent probes; provisional baseline convention |
| Initial state | Every agent starts at $\mathrm{Beta}(2,2)$ | Assistant-proposed symmetric diagnostic initialization; provisional |
| Production opportunity | Every agent is called once per round | Fixed scheduler rule; posting behavior remains inside message production |
| Posting probability | $\rho_{i,t}=1$ | Provisional null-baseline value; the interface also records no-post outcomes |
| Public message | One binary stance $M_{i,t}\in\{-1,+1\}$ when a post occurs | Retained from the parent formation probe; provisional baseline convention |
| Stance probability | $\Pr(M_{i,t}=+1)=\Pr(P_{i,t}>0.5)$ | Retained posterior-mass production rule; not psychologically validated |
| Network | Complete directed eligibility without self-links | Assistant-formalized representation of the all-other-posts boundary |
| Selection | Consumer $i$ receives every eligible $M_{j,t}$ with $j\ne i$ | Researcher-confirmed all-posts boundary, interpreted as excluding self |
| Aggregation | Raw counts plus homogeneous evidence weight | Retained baseline rule; source identity remains available in exposures |
| Evidence weight | $\eta=0.1$ | Assistant-proposed diagnostic value; provisional and uncalibrated |
| Network update | Identity: $G_{t+1}=G_t$ | Explicit null mechanism |
| Schedule | Produce all, expose and aggregate all, propose all, commit all | Synchronous implementation of the researcher-confirmed step order |
| Randomness | Separate posting, stance, selection, and network streams | Implementation convention for reproducibility and scenario isolation |

**Generated inputs.** Initial states and message draws are synthetic. They are not observations.

**Frozen or omitted neighbors.** There is no ranking, source credibility, opinion-leader role, platform preference, action type, memory decay, or network adaptation.

**Active feedback.** Messages formed from state at round $t$ affect state at $t+1$; that state affects production in the next round.

**Broken feedbacks.** Messages do not alter their producer during the same round, exposure does not depend on prior behavior or stance, and the network remains fixed.

**Clock and stopping rule.** Discrete synchronous rounds, starting from snapshot 0 and stopping after round 10.


## 3. Expected outcomes before running

- Exactly 11 messages should be produced in each round and exactly 10 should be consumed by each agent.
- Every message should reach the ten agents other than its producer; no source should have an exposure advantage.
- Every transition should be reconstructable from the prior state and the two aggregate message counts.
- With constant $\eta=0.1$, every agent's concentration should increase by exactly one per round, from 4 initially to 14 after ten rounds.
- Reordering agent iteration should not change which message or trajectory belongs to an agent.
- Because 11 binary messages cannot split evenly, each realized round must contain a temporary supportive or opposing majority. The feedback may amplify stochastic fluctuations even though selection is neutral.

A count mismatch, self-exposure, unequal source exposure, failed hand calculation, mutation during formation, non-reproducibility, or iteration-order dependence is an implementation failure. A strong drift, convergence, or polarization in one seeded trajectory is a simulation result to inspect, not automatic evidence for or against the substantive mechanism.


In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import platform
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "src" / "opinion_model").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from opinion_model.baseline import (
    BASELINE_COMPONENTS,
    RandomStreams,
    SimulationConfig,
    aggregate_messages,
    initialize_baseline,
    propose_opinion_update,
    run_simulation,
    select_messages,
    simulation_frames,
)
from opinion_model.core import (
    AgentState,
    AggregationContext,
    BetaBelief,
    Message,
    SelectionContext,
)

RUN = {
    "framework_context": "opinion-model 0.1.0 working baseline",
    "submodel_id": "coupled-null-baseline-v0.1",
    "boundary_scenario": "11-agent all-other-posts broadcast",
    "active_feedbacks": ["state -> message -> next-round state"],
    "omitted_feedbacks": [
        "opinion-leader advantage",
        "algorithmic platform preference",
        "network adaptation",
        "memory decay",
    ],
    "python": platform.python_version(),
}

RUN


## 4. Minimal specification

For agent $i$ at round $t$, the private state is

$$P_{i,t}\sim\operatorname{Beta}(a_{i,t},b_{i,t}).$$

The derived mean, signed mean, and concentration are

$$\mu_{i,t}=\frac{a_{i,t}}{a_{i,t}+b_{i,t}},\qquad x_{i,t}=2\mu_{i,t}-1,\qquad \kappa_{i,t}=a_{i,t}+b_{i,t}.$$

Formation produces one public message from the frozen round-$t$ state:

$$\pi^+_{i,t}=\Pr(P_{i,t}>0.5),\qquad M_{i,t}=\begin{cases}+1&\text{with probability }\pi^+_{i,t},\\-1&\text{otherwise.}\end{cases}$$

The neutral boundary supplies consumer $i$ with

$$\mathcal C_{i,t}=\{M_{j,t}:j\ne i\},\qquad |\mathcal C_{i,t}|=10.$$

Let $n^+_{i,t}$ and $n^-_{i,t}$ be the supportive and opposing counts in that batch. The proposed state is

$$a^*_{i,t+1}=a_{i,t}+\eta n^+_{i,t},\qquad b^*_{i,t+1}=b_{i,t}+\eta n^-_{i,t}.$$

All proposals are committed together after every proposal has been calculated.


In [ ]:
CONFIG = SimulationConfig(
    agent_count=11,
    rounds=10,
    seed=20260903,
    initial_mean=0.5,
    initial_concentration=4.0,
    post_probability=1.0,
    evidence_weight=0.1,
    consumption_capacity=10,
    exclude_self_messages=True,
)

component_register = pd.DataFrame(
    [
        {
            "component": name,
            "implementation": getattr(BASELINE_COMPONENTS, name).__name__,
            "module": getattr(BASELINE_COMPONENTS, name).__module__,
        }
        for name in (
            "initializer",
            "message_production",
            "message_selection",
            "message_aggregation",
            "opinion_update",
            "network_update",
        )
    ]
)
display(component_register)


## 5. Hand-calculable self-exclusion and aggregation check

Construct a fixed round with eight supportive and three opposing messages. A supportive producer should consume seven supportive and three opposing messages; an opposing producer should consume eight supportive and two opposing messages. This isolates self-exclusion and aggregation before stochastic simulation.


In [ ]:
HAND_PRIOR = AgentState(BetaBelief(2.0, 2.0))
HAND_MESSAGES = tuple(
    Message(
        message_id=f"r1:a{producer_id}",
        round_index=1,
        producer_id=producer_id,
        stance=1 if producer_id < 8 else -1,
    )
    for producer_id in range(11)
)
hand_world = initialize_baseline(CONFIG, RandomStreams(CONFIG.seed).initialization())
hand_selection_context = SelectionContext(
    round_index=1,
    capacity=10,
    exclude_self_messages=True,
)
hand_aggregation_context = AggregationContext(evidence_weight=0.1)

support_batch = select_messages(
    0,
    HAND_MESSAGES,
    hand_world.network,
    hand_selection_context,
    RandomStreams(CONFIG.seed).selection(1, 0),
)
oppose_batch = select_messages(
    8,
    HAND_MESSAGES,
    hand_world.network,
    hand_selection_context,
    RandomStreams(CONFIG.seed).selection(1, 8),
)
support_evidence = aggregate_messages(support_batch, hand_aggregation_context)
oppose_evidence = aggregate_messages(oppose_batch, hand_aggregation_context)
support_after = propose_opinion_update(HAND_PRIOR, support_evidence)
oppose_after = propose_opinion_update(HAND_PRIOR, oppose_evidence)

assert (support_evidence.n_support, support_evidence.n_oppose) == (7, 3)
assert (oppose_evidence.n_support, oppose_evidence.n_oppose) == (8, 2)
assert np.isclose(support_after.belief.a, 2.7)
assert np.isclose(support_after.belief.b, 2.3)
assert np.isclose(oppose_after.belief.a, 2.8)
assert np.isclose(oppose_after.belief.b, 2.2)

hand_check = pd.DataFrame(
    [
        {
            "producer_stance": "support (+1)",
            "consumed_support": support_evidence.n_support,
            "consumed_oppose": support_evidence.n_oppose,
            "posterior_a": support_after.belief.a,
            "posterior_b": support_after.belief.b,
            "posterior_mean": support_after.belief.mean,
        },
        {
            "producer_stance": "oppose (-1)",
            "consumed_support": oppose_evidence.n_support,
            "consumed_oppose": oppose_evidence.n_oppose,
            "posterior_a": oppose_after.belief.a,
            "posterior_b": oppose_after.belief.b,
            "posterior_mean": oppose_after.belief.mean,
        },
    ]
)
display(hand_check.round(4))


## 6. Package-backed synchronous round

The imported `run_simulation` scheduler calls the six registered components in a fixed order. It first forms the complete set of `ProductionOutcome` records from an immutable state snapshot. It then constructs every exposure batch, aggregate, opinion proposal, and static-network proposal. Only after all proposals exist does it create the next `WorldState`. Raw typed records are converted to data frames by the read-only observation layer.


In [ ]:
interface_summary = pd.DataFrame(
    [
        ("initialization", "SimulationConfig + initialization RNG", "WorldState at round 0"),
        ("message production", "AgentState + production context + independent RNGs", "ProductionOutcome"),
        ("message selection", "consumer + message pool + network + selection context", "Exposure batch"),
        ("message aggregation", "Exposure batch + evidence weight", "MessageEvidence"),
        ("opinion update", "prior AgentState + MessageEvidence", "proposed AgentState"),
        ("network update", "network + snapshot + round events", "proposed NetworkState"),
    ],
    columns=["component", "reads", "returns"],
)
display(interface_summary)


## 7. One seeded ten-round toy experiment

This run imports the package implementation and uses one fixed seed and one provisional parameter configuration. It is sufficient for schedule and interface inspection, not for distributional claims. Raw production opportunities, messages, exposures, aggregates, states, and network edges are retained so every displayed result can be reconstructed.


In [ ]:
SIMULATION = run_simulation(CONFIG, BASELINE_COMPONENTS)
RESULTS = simulation_frames(SIMULATION)

state_history = RESULTS["states"]
production_history = RESULTS["production"]
message_history = RESULTS["messages"]
exposure_history = RESULTS["exposures"]
aggregate_history = RESULTS["aggregates"]
network_history = RESULTS["network"]

round_messages = (
    message_history.assign(is_support=message_history["stance"].eq(1).astype(int))
    .groupby("round", as_index=False)
    .agg(n_support_messages=("is_support", "sum"))
)
round_messages["n_oppose_messages"] = CONFIG.agent_count - round_messages["n_support_messages"]
round_messages["support_share"] = round_messages["n_support_messages"] / CONFIG.agent_count

round_states = (
    state_history.groupby("round", as_index=False)
    .agg(
        population_signed_mean=("signed_mean", "mean"),
        minimum_signed_mean=("signed_mean", "min"),
        maximum_signed_mean=("signed_mean", "max"),
        concentration=("concentration", "first"),
    )
)
round_summary = round_states.merge(round_messages, on="round", how="left")

run_manifest = pd.DataFrame(
    {
        "setting": [
            "agents",
            "rounds",
            "initial Beta state",
            "posting probability",
            "evidence weight per message",
            "messages produced per round",
            "messages consumed per agent-round",
            "seed",
        ],
        "value": [
            CONFIG.agent_count,
            CONFIG.rounds,
            f"Beta({CONFIG.initial_mean * CONFIG.initial_concentration:g}, {(1 - CONFIG.initial_mean) * CONFIG.initial_concentration:g})",
            CONFIG.post_probability,
            CONFIG.evidence_weight,
            CONFIG.agent_count,
            CONFIG.consumption_capacity,
            CONFIG.seed,
        ],
    }
)

display(run_manifest)
display(round_summary.round(4))


## 8. Diagnostic views

The first view records each realized message majority. The second shows the population mean and individual range without hiding divergence. The third preserves agent-level paths. These are descriptive diagnostics for this seed, not confirmatory statistics.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)

axes[0].bar(
    round_messages["round"],
    round_messages["n_support_messages"],
    label="supportive messages",
)
axes[0].axhline(CONFIG.agent_count / 2.0, color="black", linewidth=1.0, alpha=0.5)
axes[0].set(
    title="Realized message composition",
    xlabel="round",
    ylabel="supportive messages out of 11",
    xticks=range(1, CONFIG.rounds + 1),
    ylim=(0, CONFIG.agent_count),
)

axes[1].plot(
    round_states["round"],
    round_states["population_signed_mean"],
    marker="o",
    label="population mean",
)
axes[1].fill_between(
    round_states["round"],
    round_states["minimum_signed_mean"],
    round_states["maximum_signed_mean"],
    alpha=0.2,
    label="individual range",
)
axes[1].axhline(0.0, color="black", linewidth=1.0, alpha=0.5)
axes[1].set(
    title="Population attitude path",
    xlabel="state snapshot after round",
    ylabel="signed mean",
    xticks=range(0, CONFIG.rounds + 1),
    ylim=(-1, 1),
)
axes[1].legend(fontsize=8)

agent_round_matrix = state_history.pivot(
    index="agent_id", columns="round", values="signed_mean"
).sort_index()
image = axes[2].imshow(
    agent_round_matrix.to_numpy(),
    aspect="auto",
    cmap="coolwarm",
    vmin=-1.0,
    vmax=1.0,
)
axes[2].set_xticks(range(CONFIG.rounds + 1), range(CONFIG.rounds + 1))
axes[2].set_yticks(range(CONFIG.agent_count), agent_round_matrix.index)
axes[2].set(
    title="Agent-level signed attitudes",
    xlabel="state snapshot after round",
    ylabel="agent ID",
)
colorbar = fig.colorbar(image, ax=axes[2], shrink=0.85)
colorbar.set_label("signed mean")

plt.show()

final_states = state_history[state_history["round"] == CONFIG.rounds][
    ["agent_id", "a", "b", "mean", "signed_mean", "concentration", "p_support_message"]
].reset_index(drop=True)
display(final_states.round(4))


## 9. Package integration and scientific-boundary checks

The package unit suite owns detailed rule and boundary verification. The checks below keep the notebook's hand calculation, event reconciliation, null exposure, no-learning behavior, posting-opportunity semantics, reproducibility, and order invariance visible at the experiment entry point. They establish software behavior only; they do not validate the Beta mechanism or its parameters.


In [ ]:
def canonical(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    return frame.sort_values(columns).reset_index(drop=True)


verification_rows = []

def verify(check_id: str, purpose: str, condition: bool) -> None:
    if not bool(condition):
        raise AssertionError(f"{check_id} failed: {purpose}")
    verification_rows.append(
        {"check_id": check_id, "purpose": purpose, "status": "passed"}
    )


verify(
    "V0-HAND",
    "fixed 8/3 self-exclusion case matches hand calculation",
    (support_evidence.n_support, support_evidence.n_oppose) == (7, 3)
    and (oppose_evidence.n_support, oppose_evidence.n_oppose) == (8, 2)
    and np.isclose(support_after.belief.a, 2.7)
    and np.isclose(oppose_after.belief.b, 2.2),
)
verify(
    "V2-EVENT-COUNTS",
    "production, message, exposure, aggregate, and state records are complete",
    len(production_history) == 110
    and len(message_history) == 110
    and len(exposure_history) == 1_100
    and len(aggregate_history) == 110
    and len(state_history) == 121,
)
verify(
    "V2-NULL-EXPOSURE",
    "every source reaches ten other agents without self-exposure",
    (exposure_history["consumer_id"] != exposure_history["producer_id"]).all()
    and exposure_history.groupby(["round", "consumer_id"]).size().eq(10).all()
    and exposure_history.groupby(["round", "producer_id"]).size().eq(10).all(),
)
verify(
    "V0-TRANSITIONS",
    "weighted evidence reconstructs every Beta transition",
    np.allclose(
        aggregate_history["a_after"],
        aggregate_history["a_before"] + aggregate_history["weighted_support"],
    )
    and np.allclose(
        aggregate_history["b_after"],
        aggregate_history["b_before"] + aggregate_history["weighted_oppose"],
    ),
)

no_learning = simulation_frames(
    run_simulation(replace(CONFIG, rounds=3, evidence_weight=0.0))
)
verify(
    "V0-NO-LEARNING",
    "zero evidence weight leaves beliefs unchanged",
    np.allclose(no_learning["states"]["a"], 2.0)
    and np.allclose(no_learning["states"]["b"], 2.0),
)

no_post = simulation_frames(
    run_simulation(replace(CONFIG, rounds=2, post_probability=0.0))
)
verify(
    "V0-NO-POST",
    "production opportunities remain recorded when no messages are produced",
    len(no_post["production"]) == 22
    and not no_post["production"]["did_post"].any()
    and no_post["messages"].empty
    and no_post["aggregates"]["consumed_total"].eq(0).all(),
)

repeated = simulation_frames(run_simulation(CONFIG))
for table_name in RESULTS:
    pd.testing.assert_frame_equal(RESULTS[table_name], repeated[table_name])
verification_rows.append(
    {
        "check_id": "V0-REPRODUCIBILITY",
        "purpose": "same configuration and seed reproduce every retained table",
        "status": "passed",
    }
)

reverse_order = tuple(reversed(range(CONFIG.agent_count)))
reversed_results = simulation_frames(
    run_simulation(CONFIG, agent_order=reverse_order)
)
sort_keys = {
    "states": ["round", "agent_id"],
    "production": ["round", "agent_id"],
    "messages": ["round", "producer_id"],
    "exposures": ["round", "consumer_id", "producer_id"],
    "aggregates": ["round", "consumer_id"],
    "network": ["round", "consumer_id", "producer_id"],
}
for table_name, keys in sort_keys.items():
    pd.testing.assert_frame_equal(
        canonical(RESULTS[table_name], keys),
        canonical(reversed_results[table_name], keys),
    )
verification_rows.append(
    {
        "check_id": "V2-ORDER",
        "purpose": "reversing agent iteration preserves agent-associated events and states",
        "status": "passed",
    }
)

expected_network_edges = CONFIG.agent_count * (CONFIG.agent_count - 1)
verify(
    "V2-STATIC-NETWORK",
    "the complete no-self-link network is retained at every snapshot",
    network_history.groupby("round").size().eq(expected_network_edges).all(),
)

VERIFICATION_RESULTS = pd.DataFrame(verification_rows)
display(VERIFICATION_RESULTS)
print(f"All {len(VERIFICATION_RESULTS)} package-integration checks passed.")


## 10. Observed result for the retained seed

The following compact record is generated from the event and state histories. It reports what occurred in this one synthetic trajectory without promoting that realization into a general population claim.


In [ ]:
final_population = state_history[state_history["round"] == CONFIG.rounds]
support_sequence = round_messages["n_support_messages"].astype(int).tolist()
observed_result = pd.DataFrame(
    {
        "quantity": [
            "supportive-message counts by round",
            "initial population signed mean",
            "final population signed mean",
            "final individual signed-mean range",
            "final common concentration",
            "agents with positive final signed mean",
            "highest completed software evidence level",
        ],
        "observed_value": [
            str(support_sequence),
            f"{round_states.loc[round_states['round'].eq(0), 'population_signed_mean'].iloc[0]:.4f}",
            f"{final_population['signed_mean'].mean():.4f}",
            f"[{final_population['signed_mean'].min():.4f}, {final_population['signed_mean'].max():.4f}]",
            f"{final_population['concentration'].iloc[0]:.4f}",
            int(final_population["signed_mean"].gt(0.0).sum()),
            "V2 paired coupling under the declared synthetic null boundary",
        ],
    }
)
display(observed_result)


## 11. Conditional interpretation and disposition

- Passing the checks establishes that the extracted production, neutral exposure, aggregation, opinion effect, static-network proposal, observation, and synchronous scheduler are compatible under this synthetic boundary.
- The message-count sequence and attitude path above describe one seeded realization. They do not estimate an expected trajectory or the probability of symmetry breaking.
- Equal exposure verifies the absence of source-selection advantage in this boundary. It does not test opinion-leader or platform mechanisms, which are intentionally absent.
- With 11 binary producers, every round necessarily has a realized majority. Feedback from that majority is a structural consequence of the odd population and shared boundary, not evidence of algorithmic bias.
- Constant positive $\eta$ causes concentration to accumulate. This notebook does not decide whether long-run memory decay, bounded evidence, or another update mechanism is needed.
- The initial $\mathrm{Beta}(2,2)$ state, posting probability one, $\eta=0.1$, and posterior-mass stance rule remain provisional modeling choices. Successful execution does not calibrate or empirically validate them.

**Disposition:** retain as a package-backed coupled null benchmark for researcher inspection. The scientific implementation now has replaceable component interfaces, while this notebook remains the experiment and diagnostic harness. Scenario substitution remains a later decision; no leader or platform mechanism is introduced here.
